# E-Commerce Sales & Customer Analytics — Analysis Walkthrough

Run the pipeline from the repository root first. Demo data is synthetic; use the Kaggle Olist source files for the real case study.

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
import plotly.express as px
ROOT = Path.cwd()
DB = ROOT / 'data' / 'processed' / 'warehouse.sqlite'
assert DB.exists(), 'Run the warehouse build command in README.md'
with sqlite3.connect(DB) as con:
    fact = pd.read_sql_query('SELECT * FROM fact_order_items', con)
fact.head()

## 1. Validate analytical grain

The fact table is one row per order item. Distinct orders and customers must be counted by their identifiers.

In [ ]:
summary = {'order_lines': len(fact), 'distinct_orders': fact.order_id.nunique(),
           'distinct_customers': fact.customer_unique_id.nunique(),
           'gmv_brl': round(fact.gmv.sum(), 2),
           'aov_brl': round(fact.gmv.sum() / fact.order_id.nunique(), 2)}
pd.Series(summary, name='value')

## 2. Monthly sales trend

In [ ]:
monthly = (fact.groupby('year_month', as_index=False)
           .agg(gmv=('gmv', 'sum'), orders=('order_id', 'nunique'))
           .sort_values('year_month'))
px.line(monthly, x='year_month', y='gmv', markers=True, title='Monthly GMV (BRL)')

## 3. Category revenue concentration

In [ ]:
category = fact.groupby('category', as_index=False).gmv.sum().sort_values('gmv', ascending=False)
category['revenue_share'] = category.gmv / category.gmv.sum()
category['cumulative_share'] = category.revenue_share.cumsum()
category.head(20)

## 4. Delivery performance and reviews

Order-level review and delivery values repeat on each line item. Deduplicate to order grain before aggregation.

In [ ]:
order_level = fact.drop_duplicates('order_id')
delivery = (order_level.dropna(subset=['is_late', 'review_score'])
            .groupby('is_late', as_index=False)
            .agg(orders=('order_id', 'nunique'), avg_review=('review_score', 'mean')))
delivery['delivery_status'] = delivery.is_late.map({0: 'On time', 1: 'Late'})
delivery

## Interpretation checklist

- Confirm the source mode in `data/processed/quality_report.txt`.
- Recompute all findings on real Olist data before quoting them in a resume.
- Disclose the 2016–2018 period, BRL currency and metric definitions.
- Delivery/review differences are observational associations, not proof of causality.